# اليوم الأول — معالجة النصوص والترميز
## Day 1 — Text Processing & Tokenisation

**الطالبة / Studnt:** منال قيسي— Manal qaysi  


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Manal-qaysi/bayan-nlp-Manal-qaysi/blob/main/notebooks/01_text_processing_tokenization.ipynb)

> السؤال المحوري: كيف يتحول نص عربي أو إنجليزي إلى تمثيل عددي صالح للنموذج دون إفساد معناه؟
>
> Driving question: How does Arabic or English text become a model-ready numeric representation without damaging its meaning?

**بنهاية المختبر ستتمكن من / By the end, you can:**

1. فحص Unicode بدل التخمين — inspect Unicode rather than guess.
2. بناء نسختين: نص خام محفوظ ونص مهيأ للنموذج — keep raw and model-ready copies.
3. قياس **خصوبة الترميز Token Fertility** ومعدل القطع — measure fertility and truncation.
4. تحويل Token IDs إلى Embeddings بسيطة — map token IDs to simple embeddings.
5. حفظ دليل إنجاز قابل للتسليم عبر GitHub — save GitHub-ready evidence.

> 🟣 **نسخة مشروع منال قيسي (Manal-qaysi).** المختبر الأصلي من مادة الدورة، والخلايا المعنونة بـ«🟣 مشروعي» أُضيفت لمشروعي: تربط الدفتر بمستودعي وبياناته وتطبع نتائج قابلة للتتبع بصيغة `BAYAN_RESULT::…` يجمعها `scripts/collect_results.py`.
> Run with **Runtime → Restart session and run all**, then **File → Save a copy in GitHub** to `notebooks/01_text_processing_tokenization.ipynb` in `Manal-qaysi/bayan-nlp-Manal-qaysi`.

## 0) طريقة العمل / How to work

- نفّذ الخلايا بالترتيب: **Runtime → Core → Checkpoint**.
- نتيجة كل خلية موضحة بجملة تبدأ بـ **المتوقع / Expected**.
- إذا تعذر تنزيل نموذج Hugging Face، أكمل بالمرمّز المحلي؛ فهو جزء أساسي يعمل دون شبكة بعد تثبيت الحزم.
- لا تستخدم بيانات حقيقية أو أسماء أو أرقام أشخاص. البيانات أدناه اصطناعية.

**المتوقع / Expected:** في النهاية تظهر العبارة `DAY1_NOTEBOOK1_CORE=PASS`.

In [ ]:
# إعداد بيئة قابلة لإعادة التشغيل / Reproducible setup
import importlib.util
import subprocess
import sys

PINNED = {"transformers": "5.15.1", "tokenizers": "0.22.2", "spacy": "3.8.7"}
missing = [name for name in PINNED if importlib.util.find_spec(name) is None]
if missing:
    packages = [f"{name}=={PINNED[name]}" for name in missing]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

print("Python:", sys.version.split()[0])
print("Environment ready / البيئة جاهزة")

Python: 3.13.15
Environment ready / البيئة جاهزة


In [ ]:
# 🟣 مشروعي — ربط الدفتر بمستودعي: نسخة الكود والبيانات وبصماتها (SHA-256)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Manal-qaysi/bayan-nlp-Manal-qaysi"
_candidates = [Path.cwd(), Path.cwd().parent, Path("/content/bayan-nlp-Manal-qaysi")]
REPO_DIR = next((p for p in _candidates if (p / "src" / "bayan" / "project.py").is_file()), None)
if REPO_DIR is None:
    REPO_DIR = Path("/content/bayan-nlp-Manal-qaysi") if Path("/content").is_dir() else Path("bayan-nlp-Manal-qaysi").resolve()
    subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from bayan.project import data_hashes, emit_result, git_head

DATA_DIR = REPO_DIR / "data" / "sample"
REPO_SHA = git_head(REPO_DIR)
DATA_SHA256 = data_hashes(DATA_DIR)
PROVENANCE = {"repo": REPO_URL, "repo_sha": REPO_SHA}
print("REPO_DIR:", REPO_DIR)
print("REPO_SHA:", REPO_SHA)
for _name, _digest in DATA_SHA256.items():
    print(f"  {_name}  sha256={_digest[:16]}…")

REPO_DIR: /content/bayan-nlp-Manal-qaysi
REPO_SHA: 898a347f5b50ac9d6f8653c1f53085cffc3f5bcf
  bayan_day1_sample.csv  sha256=904a5e1e860f23ac…
  bayan_day2_classification.csv  sha256=c50de92fdab1aa36…
  bayan_day2_ner.jsonl  sha256=ab413f0941656abf…
  bayan_day2_qa.json  sha256=4e894757b74d09df…
  bayan_day3_arabic.csv  sha256=0a3346b6177d0c0b…
  bayan_day3_cases.csv  sha256=322867b54d1f6f35…
  bayan_day3_predictions.csv  sha256=63b4df9dab076880…
  bayan_day3_queries.jsonl  sha256=f80ebd8b25c37b33…


### لماذا نثبت نسخًا محددة؟ / Why pin versions?

لتقليل اختلاف النتائج بين أجهزة المتدربين. التثبيت مجاني ولا يحتاج اشتراكًا؛ يحتاج اتصالًا بالإنترنت أول مرة فقط. Google Colab المجاني يكفي لهذا اليوم، ولا يضمن GPU أو مدة جلسة ثابتة.

**المتوقع / Expected:** إصدار Python ثم رسالة نجاح البيئة.

In [ ]:
import html
import re
import unicodedata
from dataclasses import dataclass
from typing import Iterable

import numpy as np
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.pre_tokenizers import BertPreTokenizer
from tokenizers.processors import TemplateProcessing

SEED = 42
rng = np.random.default_rng(SEED)
samples = [
    "أهلاً وسهلاً بكم في برنامج بيان!",
    "مرحبــاً\u00a0بكم",
    "Contact us at learner@example.org",
    "للتجربة فقط: 0551234567",
    "Natural language processing connects text and models.",
]
print("Synthetic samples:", len(samples))

Synthetic samples: 5


## 1) Unicode قبل التنظيف / Unicode before cleaning

الحرف الظاهر قد يتكون من نقطة ترميز واحدة أو أكثر. لا تخلط بين:

- **Unicode code point:** رقم معياري للحرف.
- **UTF-8:** تحويل نقاط الترميز إلى bytes للتخزين والنقل.
- **Token ID:** رقم داخل قاموس مرمّز محدد، وليس رقم Unicode.

نستخدم **NFC** لتوحيد التمثيلات المتكافئة غالبًا، لكن لا نحول العربية بقوة من دون سبب متعلق بالمهمة.

In [ ]:
def inspect_unicode(text: str) -> list[dict[str, str]]:
    return [
        {"char": ch, "code_point": f"U+{ord(ch):04X}", "name": unicodedata.name(ch, "UNKNOWN")}
        for ch in text
    ]

unicode_rows = inspect_unicode("أé")
for row in unicode_rows:
    print(row)
assert unicode_rows[0]["code_point"] == "U+0623"
print("Unicode inspection=PASS")

{'char': 'أ', 'code_point': 'U+0623', 'name': 'ARABIC LETTER ALEF WITH HAMZA ABOVE'}
{'char': 'é', 'code_point': 'U+00E9', 'name': 'LATIN SMALL LETTER E WITH ACUTE'}
Unicode inspection=PASS


**المتوقع / Expected:** صف لكل حرف يحوي الشكل و`U+....` والاسم، ثم رسالة نجاح.

### قرار عربي محافظ / Conservative Arabic decision

ابدأ بأقل تعديل ممكن. إزالة التشكيل أو توحيد الألف قد يفيد البحث أو التصنيف، لكنه قد يضر مهامًا تعتمد الشكل الأصلي. لذلك نسجل الإعدادات ولا نطبقها بصمت.

In [ ]:
ARABIC_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
TATWEEL = "\u0640"
WHITESPACE = re.compile(r"\s+")
HTML_TAG = re.compile(r"<[^>]+>")
EMAIL = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
SAUDI_MOBILE = re.compile(r"(?<!\d)(?:\+?966|00966|0)?5\d{8}(?!\d)")

@dataclass(frozen=True)
class TextRecord:
    raw_text: str
    model_text: str


def mask_pii(text: str) -> str:
    return SAUDI_MOBILE.sub("<PHONE>", EMAIL.sub("<EMAIL>", text))


def prepare_text(text: str, *, remove_diacritics: bool = False, normalize_alef: bool = False) -> TextRecord:
    if not isinstance(text, str):
        raise TypeError("text must be a string")
    raw = text
    model = unicodedata.normalize("NFC", html.unescape(text))
    model = HTML_TAG.sub(" ", model).replace(TATWEEL, "")
    if remove_diacritics:
        model = ARABIC_DIACRITICS.sub("", model)
    if normalize_alef:
        model = re.sub(r"[إأآٱ]", "ا", model)
    model = WHITESPACE.sub(" ", mask_pii(model)).strip()
    return TextRecord(raw_text=raw, model_text=model)

records = [prepare_text(text) for text in samples]
for record in records:
    print({"raw": record.raw_text, "model": record.model_text})
assert records[1].raw_text != records[1].model_text
assert "learner@example.org" not in records[2].model_text
assert "0551234567" not in records[3].model_text
assert records[2].raw_text == samples[2]
print("Two-copy preprocessing contract=PASS")

{'raw': 'أهلاً وسهلاً بكم في برنامج بيان!', 'model': 'أهلاً وسهلاً بكم في برنامج بيان!'}
{'raw': 'مرحبــاً\xa0بكم', 'model': 'مرحباً بكم'}
{'raw': 'Contact us at learner@example.org', 'model': 'Contact us at <EMAIL>'}
{'raw': 'للتجربة فقط: 0551234567', 'model': 'للتجربة فقط: <PHONE>'}
{'raw': 'Natural language processing connects text and models.', 'model': 'Natural language processing connects text and models.'}
Two-copy preprocessing contract=PASS


**المتوقع / Expected:** تبقى `raw` كما دخلت، بينما تحذف الكشيدة وتوحد المسافات وتستبدل البريد والهاتف بعلامات عامة.

> **قاعدة البيانات:** النسخة الخام للتدقيق وإعادة المعالجة، والنسخة المهيأة لدخول النموذج. لا تُنشر النسخة الخام إن احتوت بيانات شخصية.

### جرّب / Try

غيّر `remove_diacritics` و`normalize_alef` على مثال اصطناعي، ثم اكتب هل التغيير يخدم مهمتك أم يخفي فرقًا مهمًا.

## 2) spaCy كمرحلة قابلة للاختبار / A testable spaCy stage

التقسيم إلى جمل لا يُنفذ بـ`split('.')`. نمرر **نسخة النموذج المحمية** إلى spaCy، ونوثق قيدًا معروفًا: الاختصارات مثل `د.` تحتاج استثناءً أو segmenter مدربًا قبل الإنتاج. هذا المثال يستخدم `sentencizer` القاعدي بلا تنزيل نموذج لغوي.

In [ ]:
import spacy

sentence_nlp = spacy.blank("xx")
sentence_nlp.add_pipe("sentencizer")

def split_model_sentences(text: str) -> list[str]:
    protected = prepare_text(text).model_text
    return [sentence.text.strip() for sentence in sentence_nlp(protected).sents if sentence.text.strip()]

sentence_examples = {
    "ar": "الخدمة جيدة. لم يصل الرمز!",
    "en": "The portal stopped. Please retry.",
}
sentence_splits = {language: split_model_sentences(text) for language, text in sentence_examples.items()}
for language, sentences in sentence_splits.items():
    print(language, sentences)

abbreviation_probe = split_model_sentences("راجع د. أحمد. ثم أعد المحاولة.")
print("Known abbreviation probe / حالة تحتاج قاعدة إضافية:", abbreviation_probe)
assert sentence_splits["ar"] == ["الخدمة جيدة.", "لم يصل الرمز!"]
assert sentence_splits["en"] == ["The portal stopped.", "Please retry."]
print("SPACY_SENTENCE_PIPELINE=PASS")

ar ['الخدمة جيدة.', 'لم يصل الرمز!']
en ['The portal stopped.', 'Please retry.']
Known abbreviation probe / حالة تحتاج قاعدة إضافية: ['راجع د.', 'أحمد.', 'ثم أعد المحاولة.']
SPACY_SENTENCE_PIPELINE=PASS


## 2) الترميز إلى كلمات فرعية / Subword tokenisation

المرمّز المحلي التالي **تعليمي ومحدود**: يوضح الرموز الخاصة و`[UNK]` وToken IDs دون تنزيل نموذج. ليس بديلًا عن مرمّز النموذج النهائي. في Explore سنقارن اختياريًا مع mBERT.

In [ ]:
special_tokens = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
known_tokens = [
    "أهلاً", "وسهلاً", "بكم", "في", "برنامج", "بيان", "!", "مرحبا",
    "Contact", "us", "at", "<", "EMAIL", ">", "Natural", "language",
    "processing", "connects", "text", "and", "models", ".",
]
vocab_tokens = special_tokens + known_tokens
vocab = {token: idx for idx, token in enumerate(vocab_tokens)}
local_tokenizer = Tokenizer(WordPiece(vocab=vocab, unk_token="[UNK]"))
local_tokenizer.pre_tokenizer = BertPreTokenizer()
local_tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]",
    special_tokens=[("[CLS]", vocab["[CLS]"]), ("[SEP]", vocab["[SEP]"])],
)
for record in records:
    encoded = local_tokenizer.encode(record.model_text)
    print(record.model_text)
    print("tokens:", encoded.tokens)
    print("ids:   ", encoded.ids)
assert local_tokenizer.encode("مرحبا بكم").tokens == ["[CLS]", "مرحبا", "بكم", "[SEP]"]
print("Local WordPiece demonstration=PASS")

أهلاً وسهلاً بكم في برنامج بيان!
tokens: ['[CLS]', 'أهلاً', 'وسهلاً', 'بكم', 'في', 'برنامج', 'بيان', '!', '[SEP]']
ids:    [2, 5, 6, 7, 8, 9, 10, 11, 3]
مرحباً بكم
tokens: ['[CLS]', '[UNK]', 'بكم', '[SEP]']
ids:    [2, 1, 7, 3]
Contact us at <EMAIL>
tokens: ['[CLS]', 'Contact', 'us', 'at', '<', 'EMAIL', '>', '[SEP]']
ids:    [2, 13, 14, 15, 16, 17, 18, 3]
للتجربة فقط: <PHONE>
tokens: ['[CLS]', '[UNK]', '[UNK]', '[UNK]', '<', '[UNK]', '>', '[SEP]']
ids:    [2, 1, 1, 1, 16, 1, 18, 3]
Natural language processing connects text and models.
tokens: ['[CLS]', 'Natural', 'language', 'processing', 'connects', 'text', 'and', 'models', '.', '[SEP]']
ids:    [2, 19, 20, 21, 22, 23, 24, 25, 26, 3]
Local WordPiece demonstration=PASS


**المتوقع / Expected:** Tokens وIDs لكل نص. ظهور `[UNK]` يعني أن قاموس هذا المرمّز التعليمي لا يعرف الجزء؛ لا يعني أن النص خاطئ.

### مقياسان قبل اختيار المرمّز / Two pre-selection metrics

\[
\text{Token Fertility}=\frac{\text{عدد الرموز الفرعية}}{\text{عدد الكلمات التقريبية}}
\]

\[
\text{Truncation Rate}=\frac{\text{النصوص التي تتجاوز الحد}}{\text{كل النصوص}}
\]

خصوبة أعلى تعني عادة تسلسلاً أطول وكلفة أكبر، لكنها ليست وحدها حكمًا على جودة النموذج.

In [ ]:
def word_count(text: str) -> int:
    return max(1, len(text.split()))


def token_fertility(tokenizer: Tokenizer, text: str) -> float:
    tokens = tokenizer.encode(text).tokens
    content = [t for t in tokens if t not in {"[CLS]", "[SEP]", "[PAD]"}]
    return len(content) / word_count(text)


def truncation_rate(tokenizer: Tokenizer, texts: Iterable[str], max_length: int) -> float:
    texts = list(texts)
    if not texts:
        raise ValueError("texts must not be empty")
    return sum(len(tokenizer.encode(t).ids) > max_length for t in texts) / len(texts)

model_texts = [record.model_text for record in records]
fertilities = [token_fertility(local_tokenizer, text) for text in model_texts]
rate_at_10 = truncation_rate(local_tokenizer, model_texts, max_length=10)
print("fertility per sample:", [round(x, 2) for x in fertilities])
print("mean fertility:", round(float(np.mean(fertilities)), 2))
print("truncation rate @10:", f"{rate_at_10:.0%}")
assert all(x > 0 for x in fertilities)
assert 0.0 <= rate_at_10 <= 1.0
print("Tokenisation metrics=PASS")

fertility per sample: [1.17, 1.0, 1.5, 2.0, 1.14]
mean fertility: 1.36
truncation rate @10: 0%
Tokenisation metrics=PASS


## 3) Padding وTruncation ثم Embeddings

- **Padding:** إضافة `[PAD]` لتوحيد طول الدفعة؛ قناع الانتباه يضع 0 عند الحشو.
- **Truncation:** قص النص عند حد؛ قد يحذف المعلومة الحاسمة.
- **Token ID:** فهرس صحيح.
- **Embedding:** متجه أعداد حقيقية يُتعلم أثناء التدريب.

المتجهات التالية عشوائية للشرح فقط وليست تضمينات لغوية مدربة.

In [ ]:
MAX_LENGTH = 12
local_tokenizer.enable_truncation(max_length=MAX_LENGTH)
local_tokenizer.enable_padding(length=MAX_LENGTH, pad_id=vocab["[PAD]"], pad_token="[PAD]")
batch = [local_tokenizer.encode(text) for text in model_texts[:2]]
input_ids = np.array([item.ids for item in batch], dtype=np.int64)
attention_mask = np.array([item.attention_mask for item in batch], dtype=np.int64)
embedding_table = rng.normal(0.0, 0.02, size=(len(vocab), 8))
embeddings = embedding_table[input_ids]
print("input_ids shape:", input_ids.shape)
print("attention_mask shape:", attention_mask.shape)
print("embeddings shape:", embeddings.shape)
print("first attention mask:", attention_mask[0].tolist())
assert input_ids.shape == (2, MAX_LENGTH)
assert attention_mask.shape == input_ids.shape
assert embeddings.shape == (2, MAX_LENGTH, 8)
print("IDs-to-embeddings pipeline=PASS")

input_ids shape: (2, 12)
attention_mask shape: (2, 12)
embeddings shape: (2, 12, 8)
first attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]
IDs-to-embeddings pipeline=PASS


**المتوقع / Expected:** الأشكال `(2, 12)` و`(2, 12, 8)` ثم رسالة نجاح. الحشو لا يحمل معنى؛ قناع الانتباه يمنع النموذج من معاملته كنص.

## Explore — مقارنة اختيارية مع mBERT

هذه الخلية تحاول تنزيل مرمّز `google-bert/bert-base-multilingual-cased` من Hugging Face. الاستخدام مجاني، لكن التنزيل يحتاج شبكة وقد يتعذر مؤقتًا. فشلها **لا يمنع اجتياز Core**.

In [ ]:
try:
    from transformers import AutoTokenizer
    hf_tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-multilingual-cased", use_fast=True)
    comparison_text = "معالجة اللغة الطبيعية مفيدة Natural language processing is useful"
    hf_tokens = hf_tokenizer.tokenize(comparison_text)
    print("mBERT tokens:", hf_tokens)
    print("mBERT token count:", len(hf_tokens))
    print("Fast tokenizer:", hf_tokenizer.is_fast)
except Exception as exc:
    print("Optional mBERT download unavailable; continue with Core.")
    print("Reason:", type(exc).__name__)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

mBERT tokens: ['مع', '##الجة', 'اللغة', 'الطبيعية', 'م', '##فيد', '##ة', 'Natural', 'language', 'processing', 'is', 'useful']
mBERT token count: 12
Fast tokenizer: True


## 🟣 مشروعي — معالجة موحدة ذات إصدار وأدلة الترميز على بيانات بيان (T1)

الخلايا التالية أضفتها لمشروعي:

1. أطبّق عقد المعالجة الموحّد `bayan-prep/1.0.0` (من `src/bayan/project.py`، ومختبَر في `tests/`) على **كل** نصوص المشروع مع الحفاظ على النسخة الأصلية للعرض، وأتحقق أن المعالجة idempotent.
2. أقيس **fertility** و**truncation** لكل لغة بمرمّزي mBERT وDistilmBERT (مرمّز نموذج التصنيف في دفتر 03).
3. أختار `MAX_LENGTH` من القياس: أصغر طول مرشح لا يقطع أي نص من نصوص المشروع — بدل قيمة 64 الثابتة. القرار موثّق في `DECISIONS.md` (D-002).

In [ ]:
# 🟣 مشروعي — تطبيق عقد المعالجة الموحّد على نصوص المشروع كلها
import csv
from collections import Counter

from bayan.project import PREPROCESSING_VERSION, prepare_model_text

def read_csv(name):
    with (DATA_DIR / name).open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))

project_texts = []
for row in read_csv("bayan_day1_sample.csv"):
    project_texts.append({"source": "day1_sample", "language": row["language"], "display_text": row["text"]})
for row in read_csv("bayan_day2_classification.csv"):
    project_texts.append({"source": "day2_classification", "language": row["language"], "display_text": row["text"]})
for row in read_csv("bayan_day3_arabic.csv"):
    if row["variant"] != "Arabizi":  # Arabizi stays in its own lane (notebook 05)
        project_texts.append({"source": "day3_arabic_" + row["variant"], "language": "ar", "display_text": row["text"]})

for item in project_texts:
    item["model_text"] = prepare_model_text(item["display_text"], item["language"])

idempotent = all(prepare_model_text(item["model_text"], item["language"]) == item["model_text"] for item in project_texts)
changed = [item for item in project_texts if item["model_text"] != item["display_text"]]
pii_probe = prepare_model_text("راسلني على learner@example.org أو 0551234567", "ar")
print("PREPROCESSING_VERSION:", PREPROCESSING_VERSION)
print("texts:", len(project_texts), dict(Counter(item["language"] for item in project_texts)))
print("changed by preprocessing:", len(changed))
for item in changed[:3]:
    print("  before:", repr(item["display_text"]), "→ after:", repr(item["model_text"]))
print("PII probe:", pii_probe)
assert idempotent, "preprocessing must be idempotent"
assert "[EMAIL]" in pii_probe and "[PHONE]" in pii_probe
print("PROJECT_PREPROCESSING=PASS")

PREPROCESSING_VERSION: bayan-prep/1.0.0
texts: 70 {'ar': 44, 'en': 26}
changed by preprocessing: 1
  before: 'الخـدمة متأخرة مرررة اليوم' → after: 'الخدمة متأخرة مرررة اليوم'
PII probe: راسلني على [EMAIL] أو [PHONE]
PROJECT_PREPROCESSING=PASS


In [ ]:
# 🟣 مشروعي — fertility وtruncation لكل لغة بمرمّزين، واختيار MAX_LENGTH من القياس
import json
from transformers import AutoTokenizer
from bayan.project import choose_max_length, length_profile

AUDIT_TOKENIZERS = {
    "mbert": "google-bert/bert-base-multilingual-cased",
    "distilmbert": "distilbert/distilbert-base-multilingual-cased",
}
CANDIDATE_LENGTHS = [16, 24, 32, 48, 64, 96, 128]
tokenizer_audit = {}
for key, model_id in AUDIT_TOKENIZERS.items():
    audit_tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    per_language, all_lengths = {}, []
    for language in ["ar", "en"]:
        texts = [item["model_text"] for item in project_texts if item["language"] == language]
        lengths = [len(audit_tokenizer(text)["input_ids"]) for text in texts]  # includes [CLS]/[SEP]
        per_language[language] = length_profile(lengths, [len(text.split()) for text in texts], CANDIDATE_LENGTHS)
        all_lengths += lengths
    tokenizer_audit[key] = {
        "model_id": model_id,
        "vocab_size": audit_tokenizer.vocab_size,
        "per_language": per_language,
        "choice": choose_max_length(all_lengths, CANDIDATE_LENGTHS),
        "example_tokens": audit_tokenizer.tokenize(project_texts[0]["model_text"]),
    }

qa_examples = json.loads((DATA_DIR / "bayan_day2_qa.json").read_text(encoding="utf-8"))["examples"]
qa_tokenizer = AutoTokenizer.from_pretrained(AUDIT_TOKENIZERS["distilmbert"], use_fast=True)
qa_pair_lengths = [len(qa_tokenizer(row["question"], row["context"])["input_ids"]) for row in qa_examples]
qa_truncation_at_96 = sum(length > 96 for length in qa_pair_lengths) / len(qa_pair_lengths)

for key, item in tokenizer_audit.items():
    print(key, "example:", item["example_tokens"])
    for language, profile in item["per_language"].items():
        print(
            f"  {language}: n={profile['n']} fertility={profile['mean_fertility']:.2f} "
            f"p95={profile['p95_tokens']:.0f} max={profile['max_tokens']} "
            f"trunc@16={profile['truncation_rate']['16']:.0%} trunc@32={profile['truncation_rate']['32']:.0%} "
            f"trunc@64={profile['truncation_rate']['64']:.0%}"
        )
CHOSEN_MAX_LENGTH = tokenizer_audit["distilmbert"]["choice"]["max_length"]
print("CHOSEN_MAX_LENGTH (distilmBERT, zero truncation on project texts):", CHOSEN_MAX_LENGTH)
print("QA question+context: max tokens =", max(qa_pair_lengths), "| truncation@96 =", qa_truncation_at_96)

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

mbert example: ['ال', '##خدمة', 'ال', '##رق', '##مية', 'سر', '##يعة', 'و', '##وا', '##ضح', '##ة']
  ar: n=44 fertility=2.42 p95=18 max=22 trunc@16=16% trunc@32=0% trunc@64=0%
  en: n=26 fertility=1.10 p95=11 max=12 trunc@16=0% trunc@32=0% trunc@64=0%
distilmbert example: ['ال', '##خدمة', 'ال', '##رق', '##مية', 'سر', '##يعة', 'و', '##وا', '##ضح', '##ة']
  ar: n=44 fertility=2.42 p95=18 max=22 trunc@16=16% trunc@32=0% trunc@64=0%
  en: n=26 fertility=1.10 p95=11 max=12 trunc@16=0% trunc@32=0% trunc@64=0%
CHOSEN_MAX_LENGTH (distilmBERT, zero truncation on project texts): 24
QA question+context: max tokens = 41 | truncation@96 = 0.0


In [ ]:
# 🟣 مشروعي — حفظ الدليل القابل للتتبع
emit_result("nb01_tokenization", {
    **PROVENANCE,
    "preprocessing_version": PREPROCESSING_VERSION,
    "texts": len(project_texts),
    "texts_by_language": dict(Counter(item["language"] for item in project_texts)),
    "changed_by_preprocessing": len(changed),
    "before_after_examples": [{"before": item["display_text"], "after": item["model_text"]} for item in changed[:2]],
    "idempotent": idempotent,
    "pii_probe": pii_probe,
    "tokenizer_audit": tokenizer_audit,
    "candidate_lengths": CANDIDATE_LENGTHS,
    "chosen_max_length": CHOSEN_MAX_LENGTH,
    "qa_pair_max_tokens": max(qa_pair_lengths),
    "qa_truncation_at_96": qa_truncation_at_96,
    "local_wordpiece_mean_fertility": float(np.mean(fertilities)),
})

BAYAN_RESULT::nb01_tokenization::{"before_after_examples":[{"after":"الخدمة متأخرة مرررة اليوم","before":"الخـدمة متأخرة مرررة اليوم"}],"candidate_lengths":[16,24,32,48,64,96,128],"changed_by_preprocessing":1,"chosen_max_length":24,"idempotent":true,"local_wordpiece_mean_fertility":1.361904761904762,"pii_probe":"راسلني على [EMAIL] أو [PHONE]","preprocessing_version":"bayan-prep/1.0.0","qa_pair_max_tokens":41,"qa_truncation_at_96":0.0,"repo":"https://github.com/Manal-qaysi/bayan-nlp-Manal-qaysi","repo_sha":"898a347f5b50ac9d6f8653c1f53085cffc3f5bcf","texts":70,"texts_by_language":{"ar":44,"en":26},"tokenizer_audit":{"distilmbert":{"choice":{"max_length":24,"rule":"smallest candidate with truncation <= 0.0","truncation_rate":0.0},"example_tokens":["ال","##خدمة","ال","##رق","##مية","سر","##يعة","و","##وا","##ضح","##ة"],"model_id":"distilbert/distilbert-base-multilingual-cased","per_language":{"ar":{"max_tokens":22,"mean_fertility":2.421969696969697,"n":44,"p50_tokens":14.0,"p95_tokens":18.

'BAYAN_RESULT::nb01_tokenization::{"before_after_examples":[{"after":"الخدمة متأخرة مرررة اليوم","before":"الخـدمة متأخرة مرررة اليوم"}],"candidate_lengths":[16,24,32,48,64,96,128],"changed_by_preprocessing":1,"chosen_max_length":24,"idempotent":true,"local_wordpiece_mean_fertility":1.361904761904762,"pii_probe":"راسلني على [EMAIL] أو [PHONE]","preprocessing_version":"bayan-prep/1.0.0","qa_pair_max_tokens":41,"qa_truncation_at_96":0.0,"repo":"https://github.com/Manal-qaysi/bayan-nlp-Manal-qaysi","repo_sha":"898a347f5b50ac9d6f8653c1f53085cffc3f5bcf","texts":70,"texts_by_language":{"ar":44,"en":26},"tokenizer_audit":{"distilmbert":{"choice":{"max_length":24,"rule":"smallest candidate with truncation <= 0.0","truncation_rate":0.0},"example_tokens":["ال","##خدمة","ال","##رق","##مية","سر","##يعة","و","##وا","##ضح","##ة"],"model_id":"distilbert/distilbert-base-multilingual-cased","per_language":{"ar":{"max_tokens":22,"mean_fertility":2.421969696969697,"n":44,"p50_tokens":14.0,"p95_tokens":18

## 4) قرار موثق / Documented decision

| البند / Criterion | ملاحظتي / My evidence |
|---|---|
| Profile المطبق | `bayan-prep/1.0.0` المحافظ (خلية 🟣 أعلاه) |
| مثال قبل/بعد | `before_after_examples` في ناتج `nb01_tokenization` |
| متوسط Token Fertility | لكل لغة ومرمّز في ناتج الخلية أعلاه، ومنقول إلى `DECISIONS.md` D-002 |
| Truncation Rate والحد | `truncation_rate` لكل طول مرشح؛ الحد المختار `CHOSEN_MAX_LENGTH` |
| خطر محتمل على العربية | إزالة التشكيل/توحيد الألف قد تدمج كلمات مختلفة؛ لذلك لا تُطبق في نسخة النموذج هنا |
| قراري والسبب | أصغر طول لا يقطع أي نص من نصوص المشروع — D-002 |

### مستويات التحدي

- **Core:** شغّل Core واشرح مثالًا قبل/بعد.
- **Explore:** قارن المحلي مع mBERT على 5 جمل اصطناعية.
- **Distinction:** قارن مرمزين مناسبين وناقش الخصوبة والقطع والزمن دون اعتبار مقياس واحد إثباتًا للأفضلية.

In [ ]:
core_checks = {
    "unicode": unicode_rows[0]["code_point"] == "U+0623",
    "spacy_sentence_pipeline": all(len(value) == 2 for value in sentence_splits.values()),
    "raw_copy_preserved": records[2].raw_text == samples[2],
    "pii_masked": "<EMAIL>" in records[2].model_text and "<PHONE>" in records[3].model_text,
    "token_metrics": all(x > 0 for x in fertilities) and 0 <= rate_at_10 <= 1,
    "embedding_shape": embeddings.shape == (2, MAX_LENGTH, 8),
    "project_preprocessing_idempotent": idempotent,
    "project_max_length_measured": CHOSEN_MAX_LENGTH in CANDIDATE_LENGTHS,
}
for name, passed in core_checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")
assert all(core_checks.values())
print("DAY1_NOTEBOOK1_CORE=PASS")

unicode: PASS
spacy_sentence_pipeline: PASS
raw_copy_preserved: PASS
pii_masked: PASS
token_metrics: PASS
embedding_shape: PASS
project_preprocessing_idempotent: PASS
project_max_length_measured: PASS
DAY1_NOTEBOOK1_CORE=PASS


## نقطة GitHub / GitHub checkpoint

1. من Colab: **File → Save a copy in GitHub**.
2. احفظ النسخة في مستودعك العام داخل `notebooks/`.
3. Commit: `feat(day1): complete preprocessing and tokenisation lab`.
4. أضف في `README.md`: Profile، الخصوبة، معدل القطع، وقرارك.
5. لا تحفظ مفاتيح أو بيانات حقيقية أو مخرجات تكشف معلومات شخصية.

**دليل الاكتمال:** `DAY1_NOTEBOOK1_CORE=PASS` + رابط Commit عام + جدول القرار.